# 03 — Knapsack + GA Engine: Model Perencana Makan

**Hybrid algorithm (Springer Soft Computing 2024):**
1. **Knapsack greedy** per meal (breakfast/lunch/dinner) — pilih makanan optimal
2. **Genetic Algorithm (deap)** untuk 7-day diversity — chromosome 21 picks

**Score function per meal (weighted scalarization):**
```
score = w1*(protein/price) + w2*(calories/price) + w3*(fiber/price) - w4*(fat/price)
```

**GA wrapper (Springer 2024 pattern):**
- Chromosome: list 21 food indices (7 hari × 3 meal)
- Fitness: total daily_score - diversity_penalty
- Operator: tournament selection, single-point crossover, 5% mutation
- 50 generations, population 30

**Target diversity score:** ≥ 0.85 (vs ~0.7 untuk knapsack-only)

In [ ]:
import pandas as pd
import numpy as np
import json
import random
from deap import base, creator, tools, algorithms

random.seed(42)
np.random.seed(42)

df_food = pd.read_parquet('output/preprocessed/food_master.parquet')
print(f'Food master: {df_food.shape}')
df_food.head(3)

In [ ]:
# ── 1. Score Function per Goal ──
GOAL_WEIGHTS = {
    'WEIGHT_LOSS': {'protein': 0.50, 'calories': 0.20, 'fiber': 0.30, 'fat': 0.20},
    'MUSCLE_GAIN': {'protein': 0.45, 'calories': 0.40, 'fiber': 0.10, 'fat': 0.05},
    'MAINTENANCE': {'protein': 0.35, 'calories': 0.35, 'fiber': 0.20, 'fat': 0.10},
    'PERFORMANCE': {'protein': 0.40, 'calories': 0.45, 'fiber': 0.10, 'fat': 0.05},
}

def compute_score(row, goal='WEIGHT_LOSS'):
    w = GOAL_WEIGHTS[goal]
    price = max(row['estimated_price_idr'], 100)
    p_r = row['protein_g'] / (price / 1000)
    c_r = row['calories_per_portion'] / (price / 1000)
    fi_r = row.get('fiber_g', 0) / (price / 1000)
    fa_r = row['fat_g'] / (price / 1000)
    return w['protein']*p_r + w['calories']*c_r + w['fiber']*fi_r - w['fat']*fa_r

df_food['score_wl'] = df_food.apply(lambda r: compute_score(r, 'WEIGHT_LOSS'), axis=1)
print('Top 10 skor WEIGHT_LOSS:')
df_food.nlargest(10, 'score_wl')[['name','category','calories_per_portion','protein_g','estimated_price_idr','score_wl']].round(2)

In [ ]:
# ── 2. Knapsack Greedy untuk 1 meal ──
def plan_one_meal(food_df, budget_idr, calorie_target,
                   goal='WEIGHT_LOSS', dietary_restrictions=None,
                   exclude_indices=None, n_items=3):
    dietary_restrictions = dietary_restrictions or []
    exclude_indices = set(exclude_indices or [])
    cal_min, cal_max = calorie_target * 0.85, calorie_target * 1.15

    df = food_df.copy()
    if 'HALAL' in dietary_restrictions:
        df = df[df['is_halal'] == True]
    if 'VEGETARIAN' in dietary_restrictions:
        df = df[df['is_vegetarian'] == True]
    if 'VEGAN' in dietary_restrictions:
        df = df[df['is_vegan'] == True]
    
    df = df[~df.index.isin(exclude_indices)]
    df['score'] = df.apply(lambda r: compute_score(r, goal), axis=1)
    df = df.sort_values('score', ascending=False)

    selected_indices, total_cal, total_price = [], 0, 0
    for idx, row in df.iterrows():
        if len(selected_indices) >= n_items: break
        if total_price + row['estimated_price_idr'] > budget_idr: continue
        if total_cal + row['calories_per_portion'] > cal_max: continue
        selected_indices.append(idx)
        total_cal += row['calories_per_portion']
        total_price += row['estimated_price_idr']
    
    return selected_indices, total_cal, total_price

# Test
indices, cal, price = plan_one_meal(
    df_food, budget_idr=20000, calorie_target=600,
    goal='WEIGHT_LOSS', dietary_restrictions=['HALAL']
)
print(f'Picked {len(indices)} items | {cal} kcal | Rp {price:,}')
for idx in indices:
    row = df_food.loc[idx]
    print(f'  - {row["name"]} | {row["calories_per_portion"]} kcal | Rp {row["estimated_price_idr"]:,.0f}')

In [ ]:
# ── 3. Knapsack-Only 7-Day Plan (Baseline) ──
MEAL_SPLIT = {'BREAKFAST': 0.30, 'LUNCH': 0.40, 'DINNER': 0.30}

def knapsack_only_plan(food_df, budget_per_day, target_calories,
                        goal, dietary_restrictions):
    plan = []
    used_indices_3day = []
    
    for day in range(1, 8):
        day_plan = {'day': day, 'meals': {}}
        day_cal, day_price = 0, 0
        recent_exclude = set()
        # Exclude staple+protein dari 2 hari terakhir
        for past_day_indices in used_indices_3day[-2:]:
            for idx in past_day_indices:
                if food_df.loc[idx, 'category'] in ['STAPLE', 'PROTEIN']:
                    recent_exclude.add(idx)
        
        day_indices = []
        for meal_type, split in MEAL_SPLIT.items():
            indices, cal, price = plan_one_meal(
                food_df, budget_idr=budget_per_day*split,
                calorie_target=target_calories*split,
                goal=goal, dietary_restrictions=dietary_restrictions,
                exclude_indices=recent_exclude, n_items=2,
            )
            day_plan['meals'][meal_type] = [
                {'idx': int(idx), 'name': food_df.loc[idx,'name'],
                 'calories': int(food_df.loc[idx,'calories_per_portion']),
                 'price_idr': int(food_df.loc[idx,'estimated_price_idr'])}
                for idx in indices
            ]
            day_cal += cal
            day_price += price
            day_indices.extend(indices)
        
        used_indices_3day.append(day_indices)
        day_plan['total_calories'] = int(day_cal)
        day_plan['total_price_idr'] = int(day_price)
        plan.append(day_plan)
    
    return plan

plan_knapsack = knapsack_only_plan(
    df_food, budget_per_day=50000, target_calories=1900,
    goal='WEIGHT_LOSS', dietary_restrictions=['HALAL']
)

# Compute diversity score
all_names = [item['name'] for day in plan_knapsack
             for items in day['meals'].values()
             for item in items]
diversity_knapsack = len(set(all_names)) / len(all_names)
print(f'Knapsack-only 7-day plan diversity: {diversity_knapsack:.3f}')

In [ ]:
# ── 4. GA Wrapper untuk 7-Day Diversity (deap) ──
# Filter halal foods saja untuk simplified pool
df_pool = df_food[df_food['is_halal'] == True].reset_index(drop=True)
df_pool['score_wl'] = df_pool.apply(lambda r: compute_score(r, 'WEIGHT_LOSS'), axis=1)
print(f'Halal pool size: {len(df_pool)}')

# ── DEAP Setup ──
# Clear creator if already exists
if hasattr(creator, 'FitnessMax'): del creator.FitnessMax
if hasattr(creator, 'Individual'): del creator.Individual

creator.create('FitnessMax', base.Fitness, weights=(1.0,))
creator.create('Individual', list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Per slot, pick from appropriate category
# Slots: 0-6=breakfast, 7-13=lunch, 14-20=dinner (per day)
# Untuk simplicity: each slot pick any halal food, GA optimizes
POOL_SIZE = len(df_pool)
CHROMO_LEN = 21  # 7 days × 3 meals

toolbox.register('food_pick', random.randint, 0, POOL_SIZE - 1)
toolbox.register('individual', tools.initRepeat, creator.Individual,
                 toolbox.food_pick, n=CHROMO_LEN)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)

def fitness(individual):
    score = sum(df_pool.iloc[idx]['score_wl'] for idx in individual)
    
    # Diversity penalty — staple/protein duplikat dalam 3 hari berurutan
    penalty = 0
    for d in range(7):
        day_indices = individual[d*3:(d+1)*3]
        prev_window = []
        for pd in range(max(0, d-2), d):
            prev_window.extend(individual[pd*3:(pd+1)*3])
        for idx in day_indices:
            cat = df_pool.iloc[idx]['category']
            if cat in ['STAPLE', 'PROTEIN'] and idx in prev_window:
                penalty += 2.0
    
    # Budget penalty
    for d in range(7):
        day_indices = individual[d*3:(d+1)*3]
        day_price = sum(df_pool.iloc[idx]['estimated_price_idr'] for idx in day_indices)
        if day_price > 50000:
            penalty += (day_price - 50000) / 1000
    
    return (score - penalty,)

toolbox.register('evaluate', fitness)
toolbox.register('mate', tools.cxOnePoint)
toolbox.register('mutate', tools.mutUniformInt, low=0, up=POOL_SIZE-1, indpb=0.05)
toolbox.register('select', tools.selTournament, tournsize=3)

print('DEAP setup selesai.')

In [ ]:
# ── 5. Run GA ──
import time

pop_size = 30
n_generations = 50

start = time.time()
pop = toolbox.population(n=pop_size)

stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register('max', np.max)
stats.register('mean', np.mean)

pop, logbook = algorithms.eaSimple(
    pop, toolbox, cxpb=0.7, mutpb=0.2, ngen=n_generations,
    stats=stats, verbose=False
)
elapsed = time.time() - start

best_ind = tools.selBest(pop, k=1)[0]
print(f'GA selesai dalam {elapsed:.1f} detik.')
print(f'Best fitness: {best_ind.fitness.values[0]:.2f}')
print(f'Best chromosome (first 10): {best_ind[:10]}')

In [ ]:
# ── 6. Convert GA Output → 7-Day Plan Structure ──
plan_ga = []
for d in range(7):
    day_indices = best_ind[d*3:(d+1)*3]
    meals = {'BREAKFAST': [], 'LUNCH': [], 'DINNER': []}
    meal_types = ['BREAKFAST', 'LUNCH', 'DINNER']
    day_cal, day_price = 0, 0
    for i, idx in enumerate(day_indices):
        row = df_pool.iloc[idx]
        meals[meal_types[i]] = [{
            'idx': int(idx),
            'name': row['name'],
            'calories': int(row['calories_per_portion']),
            'price_idr': int(row['estimated_price_idr']),
            'category': row['category'],
        }]
        day_cal += row['calories_per_portion']
        day_price += row['estimated_price_idr']
    plan_ga.append({
        'day': d+1,
        'meals': meals,
        'total_calories': int(day_cal),
        'total_price_idr': int(day_price),
    })

# Compute diversity
all_names_ga = [item['name'] for day in plan_ga
                for items in day['meals'].values()
                for item in items]
diversity_ga = len(set(all_names_ga)) / len(all_names_ga)

print(f'\n=== COMPARISON ===')
print(f'Knapsack-only diversity: {diversity_knapsack:.3f}')
print(f'Knapsack + GA diversity: {diversity_ga:.3f}')
print(f'Improvement: {(diversity_ga - diversity_knapsack)*100:+.2f} pp')
print(f'\nGA target ≥ 0.85: {"✅" if diversity_ga >= 0.85 else "⚠️"}')

print('\nGA 7-day plan:')
for d in plan_ga:
    print(f'  Day {d["day"]}: {d["total_calories"]} kcal | Rp {d["total_price_idr"]:,}')

In [ ]:
# ── 7. Save Both Plans ──
import os
os.makedirs('output/preprocessed', exist_ok=True)

with open('output/preprocessed/plan_knapsack_only.json', 'w') as f:
    json.dump(plan_knapsack, f, indent=2, ensure_ascii=False)

with open('output/preprocessed/plan_knapsack_ga.json', 'w') as f:
    json.dump(plan_ga, f, indent=2, ensure_ascii=False)

comparison = {
    'knapsack_only_diversity': round(float(diversity_knapsack), 4),
    'knapsack_ga_diversity':   round(float(diversity_ga), 4),
    'improvement_pp':          round(float((diversity_ga - diversity_knapsack)*100), 2),
    'ga_runtime_sec':          round(elapsed, 2),
    'pop_size':                pop_size,
    'n_generations':           n_generations,
    'target_diversity':        0.85,
    'pass':                    bool(diversity_ga >= 0.85),
}
with open('output/preprocessed/comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)

print('✅ Engine selesai. Output:')
print('  output/preprocessed/plan_knapsack_only.json')
print('  output/preprocessed/plan_knapsack_ga.json')
print('  output/preprocessed/comparison.json')
print(json.dumps(comparison, indent=2))